# Nested Learning for CAD — matched CADIC seed0

Đối chứng cùng study/protocol với NESTED v5.1 đã chạy. Bank mới K10000, rawViTgeometry/fullEq9, cùng backendNN tốiưu. Không dùng learned bank NESTED hoặc clone của góiCADICcost. Đọc `docs/main_spec/matched_cadic_v5_1.md` và implementation.

**Lần đầu attach đúng3inputs:** source mới `hope_cad_matched_cadic_seed0_source_r1.zip` (ZIP hoặc Kaggle giải nén), MVTec AD cũ, `vit_base_patch8_224.augreg_in21k.safetensors` cũ. Tháo source/evidence NESTED, bridge, CADICcost và pilot/W6. Source đã đóng kèm metadata manifest/runtime; không cần output/preflight khác.

GPU T4×2; kernel/session mới; `PHASE='7.1'`, `JOB='CADIC_seed0'`, `SESSION_HOURS=9.0`. Save Version → Save & Run All. Budget9h là softlimit gồm600s exportreserve, khôngETA; atomic event dài/hardplatformkill vẫn có thể làm mấtlatestoutput. Runtime mismatch thì giữlog đểreview, không tự pipupgrade. Cóheartbeat30s/progress8ảnh.

Notebook tự stage/hash trainmanifest, kiểm discarded2normalowners reference/optimized/readonlyscore, sau đó **fresh CADIC0→3599** và15finalmetrics. Từng rawfeaturehash phải khớp NESTED reference. Test/GT chỉ evaluator sau traincomplete. Không giảmảnh/K hoặc đổiscore để khớp kếtquả mongmuốn.

`paused_at_atomic_boundary`: tảiZIP/log; phiênsau thêm **latest CADIC output** thànhinputthứ4, giữsource/dataset/checkpoint và notebook, khôngattach cácphiêncũ khác. `7.1_complete_pending_review`: đã xong, gửiZIP/log đểreview. `failed_preserve_evidence`: giữevidence, không tựretry. Không dùngCADICcostclone195 đểresume.

Output `hope_cad_matched_cadic_seed0_evidence.zip` vàKagglelog lưu tại `D:/src/DATN/.agents/artifacts/runs/support_v5_1_matched_CADIC_seed0/session_01/`; cácphiêntiếp theo session_02… Metrics ở `benchmark/runs/CADIC_seed0/metrics/`. Đủ15taskstates sẽdùng cho7.2 sau này; notebooknày chưa exposematrix/seeds/VisA. Kếtquảseed0 descriptive, chưafullstudy/CI.


In [ ]:
PHASE='7.1'
JOB='CADIC_seed0'
SESSION_HOURS=9.0
SOURCE_OVERRIDE=None
RESUME_OVERRIDE=None
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

import os,sys,time,json,hashlib,zipfile,shutil
from pathlib import Path,PurePosixPath
STARTED=time.monotonic()
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
sys.dont_write_bytecode=True
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_matched_cadic_seed0_source_r1')
WORK=Path('/kaggle/working/hope_matched_cadic_seed0')
ASSETS=Path('/tmp/hope_matched_cadic_assets')
OUTPUT=Path('/kaggle/working/hope_cad_matched_cadic_seed0_evidence.zip')
print('[MATCHED CADIC BOOTSTRAP] finding hash-bound source',flush=True)
def safe(root,name):
    p=PurePosixPath(name)
    if p.is_absolute() or p.as_posix()!=name or '..' in p.parts or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    dest=root/name
    if not dest.resolve().is_relative_to(root.resolve()):raise ValueError('Source path escapes target')
    return dest
def source_manifest(p):
    if p.is_dir():return json.loads((p/'matched_cadic_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:
        ns=z.namelist()
        if len(ns)!=len(set(ns)):raise ValueError('Duplicate source entries')
        m=json.loads(z.read('matched_cadic_manifest.json'))
        if set(ns)!=set(m['files'])|{'matched_cadic_manifest.json'}:raise ValueError('Source inventory changed')
        return m
choices=[]
paths=[Path(SOURCE_OVERRIDE)] if SOURCE_OVERRIDE else [p.parent for p in INPUT.rglob('matched_cadic_manifest.json')]+list(INPUT.rglob('*source*.zip'))
for p in paths:
    try:
        m=source_manifest(p)
        if m.get('schema')=='hope_matched_cadic_seed0_source_v1':choices.append((p,m))
    except (KeyError,zipfile.BadZipFile):
        if SOURCE_OVERRIDE:raise
if not choices or len({json.dumps(m,sort_keys=True) for p,m in choices})!=1:raise ValueError('Expected ONE matched CADIC source identity; set SOURCE_OVERRIDE')
src,m=sorted(choices,key=lambda x:str(x[0]))[0]
if ROOT.exists():raise ValueError('Source root already exists; start a fresh kernel/session')
ROOT.mkdir(parents=True)
archive=zipfile.ZipFile(src) if src.is_file() else None
try:
    for name,h in m['files'].items():
        original=safe(src,name) if not archive else None
        if original and original.is_symlink():raise ValueError('Source symlink')
        raw=archive.read(name) if archive else original.read_bytes()
        if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Source SHA mismatch: '+name)
        dst=safe(ROOT,name);dst.parent.mkdir(parents=True,exist_ok=True);dst.write_bytes(raw)
finally:
    if archive:archive.close()
(ROOT/'matched_cadic_manifest.json').write_text(json.dumps(m,sort_keys=True))
sys.path.insert(0,str(ROOT));os.chdir(ROOT)
print('[MATCHED CADIC BOOTSTRAP] verified',len(m['files']),'files; frozen core plus fresh CADIC orchestration',flush=True)
from notebook_support.matched_cadic_v5_1.common import verify_source
ORCHESTRATION,REFERENCE,TRAIN_MANIFEST=verify_source()
print('[MATCHED CADIC IDENTITY]',json.dumps(ORCHESTRATION),flush=True)


In [ ]:
from notebook_support.matched_cadic_v5_1.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,job=JOB,session_hours=SESSION_HOURS,started=STARTED,
    inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),
    RESUME_OVERRIDE=RESUME_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
